In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats

**Initial data cleanup**

Rounding to two decimal places and check for missing values in _general_data_cleanup.py_

**Validation**

In [ ]:
data = pd.read_csv("Data/USO/USO_clean.csv")
data

High/low validation

In [ ]:
sum(data["high"] < data["low"])

Chronology validation

In [ ]:
sum(data["timestamp"] != data["timestamp"].sort_values())

Time jumps

In [ ]:
data["timestamp"] = pd.to_datetime(data["timestamp"])
data["time_diffs"] = data["timestamp"] - data["timestamp"].shift(1)

print(time_diffs.head())

data["time_diffs"] = data["time_diffs"].dt.total_seconds() / 3600

time_diffs = data["time_diffs"]
time_diffs[0] = 0

print(np.sort(time_diffs.unique()))

Time elapsed clusters dictionary

In [ ]:
time_elapsed_clusters = {
    **{i: "consecutive" for i in range(1,5)},
    **{i: "intraday" for i in range(8,19)},
    **{i: "one_day" for i in range(33,45)},
    **{i: "weekend" for i in range(56,69)},
    **{i: "long_break" for i in range(81,89)},
    109: "long_break",
    114: "long_break"
}

Range of data in years:

In [ ]:

(data["timestamp"][len(data["timestamp"])-1] - data["timestamp"][0]).days/365

In [ ]:
print(data.head())

Vector of shifts between close and open prices for consecutive hours

In [ ]:
data_shifted = data
data_shifted["open"] = data_shifted["open"].shift(-1)
shifts = data_shifted["open"] - data_shifted["close"]

print(shifts.head())

shifts /= data_shifted["close"]

print(shifts.head())

average_shift = shifts.abs().mean()

print("\nAverage proportional shift: ", average_shift)

print("Max proportional shift: ", shifts.abs().max())

In [ ]:
long_shifts = shifts[time_diffs > 4]
long_shifts.abs().max()

In [ ]:
big_shifts = shifts[shifts.abs() > 0.01]
len(big_shifts)

In [ ]:
time_elapsed_big_shifts = time_diffs[shifts.abs() > 0.01]

time_elapsed_small_shifts = time_diffs[shifts.abs() <= 0.01]

percentage_inclusion_in_big_shifts = (time_elapsed_big_shifts.value_counts()*100/time_elapsed_big_shifts.count()).sort_index()

time_diffs_percentage_inclusion = (time_elapsed_small_shifts.value_counts()*100/time_elapsed_small_shifts.count()).sort_index()

print("Rest of the sample:\n", time_diffs_percentage_inclusion.groupby(time_elapsed_clusters).sum())
print("Diff > 1%:\n", percentage_inclusion_in_big_shifts.groupby(time_elapsed_clusters).sum())

#time_diffs_percentage_inclusion = (time_diffs.value_counts()*100/time_diffs.count()).sort_index()

In [ ]:
time_elapsed_small_shifts.value_counts().groupby(time_elapsed_clusters).sum()

In [ ]:
time_elapsed_big_shifts.value_counts().groupby(time_elapsed_clusters).sum()

In [ ]:
stats.ks_2samp(time_elapsed_big_shifts, time_elapsed_small_shifts, alternative='two-sided', mode="auto")

In [ ]:
stats.ttest_rel(time_elapsed_small_shifts.value_counts().groupby(time_elapsed_clusters).sum()
                , time_elapsed_big_shifts.value_counts().groupby(time_elapsed_clusters).sum(), alternative='two-sided')